In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [11]:
# Install the exact matching core packages for LangChain and OpenAI text splitters
!pip install -U --force-reinstall -q openai langchain langchain-text-splitters langchain-community langchain-openai pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

**Core Key Retrieval & Client Initialization**

In [3]:
import os
from openai import OpenAI
from langchain_openai import ChatOpenAI
from kaggle_secrets import UserSecretsClient

# Fetch the secret API key safely from Kaggle's backend vault
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("OPENAI_API_KEY")

# Set the key as an environment variable so LangChain can pick it up automatically
os.environ["OPENAI_API_KEY"] = api_key

# Initialize both interfaces
client = OpenAI(api_key=api_key)
llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)

print('Initialization complete! Both OpenAI and LangChain engines are running.')

Initialization complete! Both OpenAI and LangChain engines are running.


**Lab 9 - Task 1.2 (First API Call Verification)**

In [ ]:
# Make First Verification Call using gpt-4
try:
    response = client.chat.completions.create(
        model='gpt-4',
        messages=[{'role': 'user', 'content': 'Say hello and introduce yourself briefly!'}],
        max_tokens=150,
        temperature=0.7
    )
    print(response.choices[0].message.content)
except Exception as e:
    print(f"Connection failure encountered: {e}")

**Lab 9 - Parts 2 & 3 (Interactive Domain Chatbot Engine)**

In [6]:
def start_chat_session(system_prompt):
    """Initializes a conversational multi-turn terminal framework inside your cell."""
    messages = [{'role': 'system', 'content': system_prompt}]
    print('Chatbot online! Type "quit" or "exit" to close the chat pipeline.\n')
    
    while True:
        user_input = input('You: ')
        if user_input.lower() in ['quit', 'exit']:
            print('Goodbye!')
            break
            
        if not user_input.strip():
            continue
            
        messages.append({'role': 'user', 'content': user_input})
        
        try:
            response = client.chat.completions.create(
                model='gpt-3.5-turbo',
                messages=messages,
                max_tokens=500,
                temperature=0.7
            )
            reply = response.choices[0].message.content
            messages.append({'role': 'assistant', 'content': reply})
            print(f'Assistant: {reply}\n')
        except Exception as e:
            print(f"Error connecting: {e}")

# Prompt definitions matching Lab 9 parameters
hr_prompt = "You are an HR assistant for a technology company. Policies:\nVacation: 15 days per year\nSick leave: Unlimited (with manager approval)\nRemote work: 3 days per week\nHealth insurance: Fully covered\n401(k) matching: Up to 5%\n\nRole: Answer accurately, remain supportive, keep answers within 2-3 sentences."
support_prompt = "You are a customer support agent for TechShop. Policies:\nReturns: 30-day return policy\nShipping: Free over $50, otherwise $5.99\nWarranty: 1 year manufacturer warranty\nHours: 9 AM - 6 PM EST, Mon-Fri\n\nTone: Empathetic, patient, apologize when necessary, escalate when complex."

**Lab 10 - Document Generation Script**

In [7]:
# Programmatically build directory structure and inject content files
os.makedirs('company_docs', exist_ok=True)

hr_policy_text = """Employee Handbook HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days of continuous employment.
Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires explicit manager approval.
Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers."""

with open('company_docs/hr_policy.txt', 'w') as f:
    f.write(hr_policy_text)

print("Local repository directory 'company_docs/hr_policy.txt' generated successfully.")

Local repository directory 'company_docs/hr_policy.txt' generated successfully.


**Lab 10 - Task 1.1 & 1.2 (Document Extraction & Chunking)******

Lab 10 - Task 2.1 (Keyword Metric Search Implementation)

In [19]:
import os

# 1. Read the local file directly using Python's built-in file tools
file_path = 'company_docs/hr_policy.txt'

if os.path.exists(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        file_content = f.read()
    print("Successfully loaded 1 document context object.")
else:
    file_content = ""
    print("Error: 'company_docs/hr_policy.txt' not found. Please make sure Cell 5 ran successfully!")

# 2. Replicate the text splitter manually (Chunks text roughly by paragraph segments)
# This splits text cleanly at paragraph breaks or double newlines just like LangChain does
raw_paragraphs = [p.strip() for p in file_content.split('\n\n') if p.strip()]

# Create a mock Class object so Cell 7 and Cell 8 still work perfectly without modifications
class MockChunk:
    def __init__(self, content):
        self.page_content = content

chunks = [MockChunk(p) for p in raw_paragraphs]
print(f'Successfully generated {len(chunks)} structural context chunks.')

Successfully loaded 1 document context object.
Successfully generated 1 structural context chunks.


Lab 10 - Task 3.1 & 3.2 (The Complete RAG Architecture & Benchmarking)

In [21]:
# Cell 7: Simple Keyword Search Implementation
def simple_search(query, chunks, top_k=3):
    """Scores document string intersections to find context matches."""
    query_lower = query.lower()
    scored_chunks = []
    
    for chunk in chunks:
        content_lower = chunk.page_content.lower()
        score = sum(content_lower.count(word) for word in query_lower.split())
        if score > 0:
            scored_chunks.append((score, chunk))
            
    # Sort structures via score performance metric
    scored_chunks.sort(reverse=True, key=lambda x: x[0])
    return [chunk for score, chunk in scored_chunks[:top_k]]

In [22]:
# Task 3.1 & 3.2: Pure Python RAG Pipeline Execution Loop

def rag_query(query, chunks):
    """Coordinates context gathering with final OpenAI sequence generation."""
    # 1. Retrieve the relevant text snippets from Cell 6
    relevant_chunks = simple_search(query, chunks, top_k=3)
    
    if not relevant_chunks:
        return 'No relevant information found in documents.'
        
    # 2. Combine the chunk contents together into a text string
    context = "\n\n---\n\n".join([chunk.page_content for chunk in relevant_chunks])
    
    # 3. Create the direct instruction prompt matching lab requirements
    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context provided below.
If the answer is not in the context, say so clearly.

Context:
{context}

Question: {query}
Answer:"""

    # 4. Use the stable, official client to generate the answer
    try:
        response = client.chat.completions.create(
            model='gpt-3.5-turbo',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0  # Deterministic setting for factual answers
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error gathering AI reply: {e}"

def ask_without_rag(question):
    """Direct unconditioned prompt evaluation to simulate standard GPT behavior."""
    try:
        response = client.chat.completions.create(
            model='gpt-3.5-turbo',
            messages=[
                {'role': 'system', 'content': 'You are a helpful HR assistant.'},
                {'role': 'user', 'content': question}
            ],
            temperature=0
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error gathering AI reply: {e}"

# --- Test execution evaluation run ---
test_query = "How many vacation days do full-time employees get?"

print("WITHOUT RAG (Generic Model Response):")
print(ask_without_rag(test_query))

print("\n" + "="*50 + "\n")

print("WITH RAG (Contextually Targeted System Response):")
print(rag_query(test_query, chunks))

WITHOUT RAG (Generic Model Response):
Error gathering AI reply: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


WITH RAG (Contextually Targeted System Response):
Error gathering AI reply: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
